<h1 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">مشروع الرؤية الحاسوبية والتعلّم العميق: الفرز الذكي للنفايات</h1>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>الفكرة:</strong> تحليل صورة جسم مرمي وتصنيفه إلى زجاج، ورق، كرتون، بلاستيك، معدن، أو نفايات أخرى. نستخدم OpenCV لفحص الصورة وتحسينها، ثم شبكة CNN للتعلّم والتصنيف.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>سؤال المشروع:</strong> هل يستطيع النموذج تعميم ما تعلمه من صور بخلفيات بسيطة على صور جديدة التقطناها في بيئتنا؟</p>
<blockquote dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">هذا نموذج تعليمي للتصنيف بالصورة، وليس نظامًا آليًا يتحكم بمعدات فرز حقيقية.</blockquote>

<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">خطة العمل</h2>
<ol dir="rtl" style="direction:rtl;text-align:right"><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تحميل مجموعة TrashNet مباشرة في Google Colab.</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">عرض الصور وفحصها بصريًا، ثم تطبيق تحويلات OpenCV مثل BGR إلى RGB وHSV وتحسين التباين CLAHE.</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تجهيز تقسيم تدريب/تحقق/اختبار مع مراعاة تفاوت أعداد الفئات.</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تدريب CNN صغيرة مع زيادة بيانات مناسبة، ورسم منحنيات الخسارة والدقة.</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تقييم النموذج بمصفوفة الالتباس وPrecision وRecall وF1 لكل فئة.</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تجربة صور جديدة من الهاتف ومناقشة اختلاف الخلفية والإضاءة.</li></ol>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">المجموعة الأصلية تضم 2,527 صورة في ست فئات، وصورها غالبًا ملتقطة أمام خلفية بيضاء؛ لذلك اختبار صور جديدة من الواقع يضيف جزءًا مهمًا للمشروع. <a href="https://github.com/garythung/trashnet">المصدر الأصلي للمجموعة</a></p>

In [ ]:
# هذا الدفتر مُعدّ للتشغيل في Google Colab.
import sys, os, random, zipfile, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
%pip -q install arabic-reshaper python-bidi
import arabic_reshaper
from bidi.algorithm import get_display

def rtl_text(text):
    return get_display(arabic_reshaper.reshape(str(text)))


In [ ]:
# تنزيل النسخة المصغّرة (حوالي 41 MB) وفكها تلقائيًا داخل Colab.
DATA_ROOT = Path("/content/trashnet")
DATA_DIR = DATA_ROOT / "dataset-resized"
ZIP_PATH = Path("/content/trashnet_dataset.zip")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
    urllib.request.urlretrieve(url, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)

if not DATA_DIR.exists():
    # بعض نسخ الأرشيف تضع مجلد الفئات مباشرة داخل مجلد آخر.
    candidates = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and any(x.is_dir() for x in p.iterdir())]
    DATA_DIR = next((p for p in candidates if {"glass", "paper"}.issubset({x.name.lower() for x in p.iterdir() if x.is_dir()})), DATA_DIR)
print("مجلد البيانات:", DATA_DIR)


In [ ]:
# جرد الصور والفئات
EXTS = {".jpg", ".jpeg", ".png"}
class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]
records = [(str(f), class_names.index(folder.name))
           for folder in class_dirs for f in folder.rglob("*") if f.suffix.lower() in EXTS]
if not records:
    raise FileNotFoundError("لم أجد صورًا. افحص مجلد البيانات الذي طُبع في الخلية السابقة.")
paths = np.array([r[0] for r in records])
labels = np.array([r[1] for r in records], dtype=np.int32)
print("الفئات:", class_names)
print("إجمالي الصور:", len(paths))
for name, count in zip(class_names, np.bincount(labels, minlength=len(class_names))):
    print(f"{name:12s}: {count}")


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">فحص ومعالجة الصورة بـ OpenCV</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">صور OpenCV الملونة تقرأ افتراضيًا بترتيب BGR. نعرض تحويلها إلى RGB، ونفحص تمثيل HSV، ثم نجرّب CLAHE لتحسين التباين المحلي. في التدريب سنحافظ على RGB ونستخدم تحجيمًا وزيادة بيانات؛ اعرض تأثير CLAHE وناقش إن كان مفيدًا قبل اعتماد أي تحسين في النموذج.</p>

In [ ]:
sample_path = paths[0]
bgr = cv2.imread(sample_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
hue = hsv[:, :, 0]
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(rgb); ax[0].set_title(rtl_text("RGB الأصلي"))
ax[1].imshow(hue, cmap="hsv"); ax[1].set_title(rtl_text("قناة Hue من HSV"))
ax[2].imshow(enhanced); ax[2].set_title(rtl_text("بعد CLAHE"))
for a0 in ax: a0.axis("off")
plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تقسيم البيانات وتجهيز الدفعات</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نفصل مجموعة الاختبار قبل التدريب، ولا نستخدمها لاختيار النموذج. التقسيم طبقي ليحافظ على نسبة كل فئة، ونحسب أوزان الفئات لأن عدد صور «trash» أقل من بعض الفئات الأخرى.</p>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_class_weight

train_paths, temp_paths, train_y, temp_y = train_test_split(
    paths, labels, test_size=0.30, random_state=SEED, stratify=labels)
val_paths, test_paths, val_y, test_y = train_test_split(
    temp_paths, temp_y, test_size=0.50, random_state=SEED, stratify=temp_y)
print(f"تدريب: {len(train_y)} | تحقق: {len(val_y)} | اختبار: {len(test_y)}")

IMG_SIZE = (160, 160)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

def make_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_y, True)
val_ds = make_dataset(val_paths, val_y)
test_ds = make_dataset(test_paths, test_y)
classes = np.unique(train_y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_y)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print("أوزان الفئات:", {class_names[k]: round(v, 2) for k, v in class_weights.items()})


In [ ]:
# معاينة عينات بعد التحميل
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, (image, label) in zip(axes.flat, train_ds.unbatch().take(6)):
    ax.imshow(tf.cast(image, tf.uint8).numpy())
    ax.set_title(class_names[int(label.numpy())]); ax.axis("off")
plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">بناء CNN وتدريبها</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نضيف انعكاسًا ودورانًا خفيفًا أثناء التدريب فقط. تختار EarlyStopping أفضل مرحلة اعتمادًا على مجموعة التحقق. غيّر عدد الحقب بعد أن تتأكد أن الدفتر يعمل على Colab.</p>

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.08),
], name="augmentation")

model = tf.keras.Sequential([
    tf.keras.Input(shape=(*IMG_SIZE, 3)),
    augmentation,
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
], name="waste_cnn")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=12,
                    class_weight=class_weights, callbacks=callbacks)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["loss"], label=rtl_text("تدريب")); ax[0].plot(history.history["val_loss"], label=rtl_text("تحقق"))
ax[0].set_title(rtl_text("الخسارة")); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=.2)
ax[1].plot(history.history["accuracy"], label=rtl_text("تدريب")); ax[1].plot(history.history["val_accuracy"], label=rtl_text("تحقق"))
ax[1].set_title(rtl_text("الدقة")); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تقييم الاختبار</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نستخدم الاختبار بعد التدريب فقط. راجع أداء كل فئة خصوصًا الفئة الأقل عددًا، ولا تعتمد على Accuracy وحدها.</p>

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
print("دقة الاختبار:", round(accuracy_score(test_y, y_pred), 3))
print(classification_report(test_y, y_pred, target_names=class_names, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(test_y, y_pred), display_labels=class_names).plot(
    cmap="Blues", xticks_rotation=35, values_format="d", colorbar=False)
plt.title(rtl_text("مصفوفة الالتباس — الاختبار")); plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تجربة صورة جديدة من جهازك</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">في Google Colab شغّل الخلية وارفع صورة واحدة. يطبع النموذج الفئة المتوقعة واحتمالها. جرّب صورًا ذات خلفية وإضاءة مختلفتين عن خلفية المجموعة، وسجّل الحالات التي أخطأ فيها.</p>

In [ ]:
from google.colab import files
from PIL import Image

def predict_uploaded_image(file_path):
    bgr = cv2.imread(file_path)
    if bgr is None:
        raise ValueError("تعذر فتح الصورة")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, IMG_SIZE)
    score = model.predict(resized[None, ...], verbose=0)[0]
    best = int(score.argmax())
    plt.figure(figsize=(4, 4)); plt.imshow(rgb); plt.axis("off")
    plt.title(f"{class_names[best]} | {rtl_text('الثقة')} {score[best]:.1%}"); plt.show()
    return sorted(zip(class_names, score.tolist()), key=lambda x: x[1], reverse=True)

uploaded = files.upload()
for filename in uploaded:
    print(predict_uploaded_image(filename))


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">خلاصة المشروع</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">اكتب خلاصة قصيرة تجيب عن الأسئلة التالية:</p>
<ul dir="rtl" style="direction:rtl;text-align:right"><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">ما أكثر فئتين يخلط النموذج بينهما؟</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">هل ساعدت أوزان الفئات في تحسين الفئة الأقل عددًا؟</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">هل تغيرت النتيجة على الصور التي التقطتها بنفسك؟ وما سبب ذلك؟</li><li dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">ما حدود مجموعة البيانات؟</li></ul>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>تنبيه:</strong> مجموعة TrashNet صغيرة وصورها غالبًا على خلفية بيضاء، ووسمها «trash» ليس مرادفًا لكل أنواع النفايات العضوية. نتائج النموذج لا تضمن فرزًا صحيحًا في كل البيئات. احفظ النموذج بعد انتهاء التجربة باستخدام <code dir="ltr" style="direction:ltr; unicode-bidi:isolate">model.save(&quot;waste_classifier.keras&quot;)</code>.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>الاستشهاد:</strong> Thung, G. &amp; Yang, M. (2016). <em>Classification of Trash for Recyclability Status</em>. Stanford CS229 project. Dataset repository: https://github.com/garythung/trashnet</p>